# cosine_skip PYNQ test

Copy `cosine_overlay.bit` and `cosine_overlay.hwh` into the same folder as this notebook before running it on PYNQ-Z2.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

overlay = Overlay('cosine_overlay.bit')
ip = overlay.cosine_skip_0
print(ip.register_map)

In [ ]:
# HLS register map offsets from cosine_overlay.hwh
CTRL = 0x00
AP_RETURN = 0x10
X_LOW = 0x18
X_HIGH = 0x1C
Y_LOW = 0x24
Y_HIGH = 0x28
LENGTH = 0x30
THRESHOLD_Q15 = 0x38
DOT_OUT_LOW = 0x40
NORM_X_LOW = 0x58
NORM_Y_LOW = 0x70

def write_u64(offset_low, value):
    value = int(value)
    ip.write(offset_low, value & 0xffffffff)
    ip.write(offset_low + 4, (value >> 32) & 0xffffffff)

def read_u64(offset_low):
    return ip.read(offset_low) | (ip.read(offset_low + 4) << 32)

def read_i64(offset_low):
    value = read_u64(offset_low)
    return value - (1 << 64) if value & (1 << 63) else value

def run_cosine_skip(x_buf, y_buf, threshold=0.999):
    threshold_q15 = int(threshold * 32768)
    x_buf.flush()
    y_buf.flush()
    write_u64(X_LOW, x_buf.physical_address)
    write_u64(Y_LOW, y_buf.physical_address)
    ip.write(LENGTH, len(x_buf))
    ip.write(THRESHOLD_Q15, threshold_q15)
    t0 = time.time()
    ip.write(CTRL, 0x01)
    while (ip.read(CTRL) & 0x2) == 0:
        pass
    t1 = time.time()
    decision = ip.read(AP_RETURN)
    dot = read_i64(DOT_OUT_LOW)
    norm_x = read_u64(NORM_X_LOW)
    norm_y = read_u64(NORM_Y_LOW)
    similarity = dot / np.sqrt(norm_x * norm_y) if norm_x and norm_y else np.nan
    return decision, similarity, (t1 - t0) * 1000

In [ ]:
N = 32768
x = allocate(shape=(N,), dtype=np.int16)
y = allocate(shape=(N,), dtype=np.int16)
rng = np.random.default_rng(1234)

x[:] = rng.integers(-2048, 2048, size=N, dtype=np.int16)
y[:] = x[:]
result, similarity, elapsed_ms = run_cosine_skip(x, y)
print('same vectors: should_skip =', int(result), 'similarity =', similarity, 'elapsed ms =', elapsed_ms)

y[:] = rng.integers(-2048, 2048, size=N, dtype=np.int16)
result, similarity, elapsed_ms = run_cosine_skip(x, y)
print('different vectors: should_skip =', int(result), 'similarity =', similarity, 'elapsed ms =', elapsed_ms)